# Clase · Siniestros de tránsito: Región Metropolitana y La Araucanía
**Módulo 3 · Aplicaciones de Ciencia de Datos Espaciales en Estudios Urbanos** · Diplomado en Análisis Espacial de Datos (FAU, U. de Chile)

Este notebook trabaja con la capa de **siniestros individuales** de CONASET (Observatorio de Seguridad Vial). Lo recorremos primero completo con la **Región Metropolitana**, paso a paso y con código simple (casi sin funciones propias). Después vienen tramos marcados como **"Ahora ustedes"**, donde ustedes repiten el mismo trabajo con **La Araucanía**, escribiendo el código sin una plantilla resuelta.

No hace falta ser una experta o experto en Python: casi todo el notebook son líneas de pandas encadenadas, una debajo de la otra. Cuando algo se repite mucho sí usamos una función corta (ancho de banda del mapa de calor, por ejemplo), pero son la excepción, no la norma.

**Descarga previa:** desde `mapas-conaset.opendata.arcgis.com` bajen la capa de **siniestros individuales** de la Región Metropolitana y de La Araucanía, para el mismo año, y guárdenla en la carpeta `datos/` (CSV, GeoJSON o Shapefile).

In [1]:
# En Colab, descomenten esta línea la primera vez (tarda ~1 minuto)
# !pip install geopandas esda libpysal folium mapclassify seaborn --quiet

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Rutas y parámetros: lo único que deben cambiar para que el notebook corra en su computador
RUTA_RM = 'datos/siniestros_rm.csv'          # <- cambien por su archivo de la Región Metropolitana
RUTA_ARAUCANIA = 'datos/siniestros_araucania.csv'   # <- cambien por su archivo de La Araucanía
ANIO = 2025

CRS_GRADOS = 'EPSG:4326'   # cómo vienen las coordenadas (WGS84)
CRS_METROS_RM = 'EPSG:32719'         # UTM 19S, para medir distancias en la RM
CRS_METROS_ARAUCANIA = 'EPSG:32718'  # UTM 18S, para medir distancias en La Araucanía

# Cifras oficiales 2025 para validar (CONASET, Informe Nacional 2025, Tabla 5)
REF_RM = {'siniestros': 20291, 'fallecidos': 350, 'lesionados': 8017}
REF_ARAUCANIA = {'siniestros': 5587, 'fallecidos': 106, 'lesionados': 4011}

### Si no pueden descargar el archivo
Ejecuten esta celda solo si no tienen el CSV real. Genera una tabla de práctica con problemas típicos (coma decimal, coordenadas en cero o invertidas). **No es un dato real**, solo sirve para poder seguir la clase.

## 1. Región Metropolitana: cargar y mirar la tabla
Leemos **todo como texto** (`dtype=str`). Así ninguna columna se rompe al cargar —por ejemplo, coordenadas con coma decimal— y convertimos a número nosotros, con control.

In [ ]:
rm = pd.read_csv(RUTA_RM, dtype=str)
print(rm.shape)
rm.head(3)

In [ ]:
rm.columns.tolist()

In [ ]:
# Si los nombres de columna de su archivo son distintos, renombren aquí.
# El lado izquierdo es el nombre que usaremos en todo el notebook; el derecho, el de su archivo.
rm = rm.rename(columns={
    'FECHA': 'fecha', 'HORA': 'hora', 'COMUNA': 'comuna', 'TIPO': 'tipo',
    'FALLECIDOS': 'fallecidos', 'GRAVES': 'graves', 'LATITUD': 'lat', 'LONGITUD': 'lon',
})
rm.columns.tolist()

## 2. Región Metropolitana: limpiar coordenadas y filas repetidas

In [ ]:
# 2.1 Coma decimal -> punto, y texto -> número
rm['lat'] = pd.to_numeric(rm['lat'].str.replace(',', '.', regex=False), errors='coerce')
rm['lon'] = pd.to_numeric(rm['lon'].str.replace(',', '.', regex=False), errors='coerce')
rm[['lat', 'lon']].describe()

In [ ]:
# 2.2 ¿Cuántas coordenadas quedaron nulas o en cero?
print('Nulas:', rm['lat'].isna().sum())
print('En cero:', ((rm['lat'] == 0) | (rm['lon'] == 0)).sum())

In [ ]:
# 2.3 Nos quedamos solo con coordenadas dentro de un rango plausible para Chile continental
# (latitud entre -56 y -17; longitud entre -76 y -66; fuera de ese rango descartamos)
rm = rm[rm['lat'].between(-56, -17) & rm['lon'].between(-76, -66)].copy()
print('Filas después del filtro de coordenadas:', len(rm))

In [ ]:
# 2.4 Fechas, hora y columnas numéricas
rm['fecha'] = pd.to_datetime(rm['fecha'], errors='coerce')
rm['hora'] = pd.to_numeric(rm['hora'], errors='coerce')
rm['fallecidos'] = pd.to_numeric(rm['fallecidos'], errors='coerce').fillna(0)
rm['graves'] = pd.to_numeric(rm['graves'], errors='coerce').fillna(0)

# 2.5 Filas idénticas (por si la capa trae duplicados)
duplicadas = rm.duplicated().sum()
print('Filas exactamente duplicadas:', duplicadas)
rm = rm.drop_duplicates()
print('Filas finales, RM:', len(rm))

## 3. Región Metropolitana: validar contra la cifra oficial
Comparamos lo que tenemos contra el Informe Nacional 2025 de CONASET (Tabla 5, nivel regional).

In [ ]:
mis_siniestros = len(rm)
mis_fallecidos = int(rm['fallecidos'].sum())

print('Mis siniestros:', mis_siniestros, ' vs. CONASET:', REF_RM['siniestros'])
print('Mis fallecidos :', mis_fallecidos, ' vs. CONASET:', REF_RM['fallecidos'])
print('Diferencia en siniestros: {:.1f}%'.format((mis_siniestros / REF_RM['siniestros'] - 1) * 100))

Con datos reales, una diferencia de más o menos un pequeño porcentaje es normal (fecha de corte de la descarga, alguna deduplicación adicional). Una diferencia enorme (la mitad, el doble) avisa de un problema de carga o de filtro. Con los datos de práctica esta comparación no va a calzar: es esperable.

## 4. Región Metropolitana: indicadores por comuna

In [ ]:
por_comuna = rm.groupby('comuna').agg(
    siniestros=('fallecidos', 'size'),
    fallecidos=('fallecidos', 'sum'),
    graves=('graves', 'sum'),
).reset_index()

por_comuna['severidad'] = (por_comuna['fallecidos'] / por_comuna['siniestros'] * 100).round(2)
por_comuna = por_comuna.sort_values('siniestros', ascending=False)
por_comuna.head(10)

In [ ]:
# Volumen vs. severidad: comunas con muchos siniestros no son necesariamente las más letales
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=por_comuna, x='siniestros', y='severidad', size='fallecidos', sizes=(30, 300), ax=ax, legend=False)
for _, fila in por_comuna.nlargest(6, 'severidad').iterrows():
    ax.annotate(fila['comuna'], (fila['siniestros'], fila['severidad']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_title('Región Metropolitana: volumen vs. severidad por comuna')
ax.set_xlabel('siniestros'); ax.set_ylabel('fallecidos por 100 siniestros')
plt.tight_layout(); plt.show()

## 5. Región Metropolitana: patrón por hora del día

In [ ]:
por_hora = rm.dropna(subset=['hora']).groupby(rm['hora'].astype(int)).size()
por_hora = por_hora.reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(8, 3.5))
por_hora.plot(kind='bar', ax=ax, color='#1D4F91')
ax.set_title('Región Metropolitana: siniestros por hora del día')
ax.set_xlabel('hora'); ax.set_ylabel('siniestros')
plt.tight_layout(); plt.show()

---
## 6. Ahora ustedes: repitan los pasos 1 a 5, pero con La Araucanía

No hay una plantilla resuelta en esta sección: escriban ustedes el código, mirando lo que hicimos arriba para la Región Metropolitana como referencia. Las celdas siguientes solo traen el objetivo de cada paso, en un comentario. Si se atascan en un paso, revisen la celda equivalente del bloque de la RM: la lógica es la misma, solo cambia el nombre de la tabla.

In [ ]:
# 6.1 Carguen el archivo de La Araucanía (RUTA_ARAUCANIA) con dtype=str y muestren las primeras filas.
# Si los nombres de columna no calzan con los que usamos para la RM, renómbrenlos aquí también.

# TU CÓDIGO AQUÍ
araucania = None  # reemplacen esta línea

In [ ]:
# 6.2 Limpien 'lat' y 'lon': de texto con coma a número con punto, y filtren fuera de rango
# (usen el mismo rango de la RM: latitud entre -56 y -17, longitud entre -76 y -66).

# TU CÓDIGO AQUÍ

In [ ]:
# 6.3 Conviertan 'fecha' a fecha, 'hora' a número, y 'fallecidos'/'graves' a número (nulos -> 0).
# Eliminen filas exactamente duplicadas. Impriman cuántas filas quedaron.

# TU CÓDIGO AQUÍ

In [ ]:
# 6.4 Comparen sus totales (siniestros y fallecidos) contra REF_ARAUCANIA e impriman la diferencia
# porcentual en siniestros, igual que hicimos para la RM.

# TU CÓDIGO AQUÍ

In [ ]:
# 6.5 Calculen la tabla por_comuna_araucania: siniestros, fallecidos, graves y severidad por comuna,
# ordenada de mayor a menor número de siniestros. Muestren las primeras 10 filas.

# TU CÓDIGO AQUÍ

In [ ]:
# 6.6 Grafiquen siniestros por hora del día para La Araucanía (igual que en el paso 5 de la RM).
# ¿La forma del gráfico se parece a la de la Región Metropolitana? Coméntenlo en una celda de texto.

# TU CÓDIGO AQUÍ

## 7. Geometría y mapa (Región Metropolitana)
Con `lat` y `lon` ya limpias, construimos los puntos y los llevamos a un sistema de coordenadas en metros para poder medir distancias.

In [ ]:
rm_geo = gpd.GeoDataFrame(rm, geometry=gpd.points_from_xy(rm['lon'], rm['lat']), crs=CRS_GRADOS)
rm_utm = rm_geo.to_crs(CRS_METROS_RM)

print('CRS original   :', rm_geo.crs)
print('CRS proyectado  :', rm_utm.crs)
rm_utm[['comuna', 'tipo', 'fallecidos']].head(3)

In [ ]:
# Mapa interactivo (requiere folium y mapclassify)
rm_geo.explore(column='tipo', tiles='CartoDB positron', marker_kwds={'radius': 3}, legend=True)

## 8. Mapa de calor (KDE) y puntos calientes (Región Metropolitana)
Usamos una sola función corta para el suavizado, porque construir la grilla a mano es repetitivo. Todo lo demás sigue siendo código directo.

In [ ]:
def suavizar(x, y, celda=200, ancho=600, margen=1500):
    # Cuenta puntos en una grilla fina y los suaviza con un promedio móvil (equivalente simple a un KDE).
    # celda: tamaño de cada casillero (metros). ancho: cuánto "difumina" cada punto (metros).
    from scipy.ndimage import uniform_filter
    xe = np.arange(x.min() - margen, x.max() + margen, celda)
    ye = np.arange(y.min() - margen, y.max() + margen, celda)
    grilla, _, _ = np.histogram2d(x, y, bins=[xe, ye])
    suave = uniform_filter(grilla, size=int(ancho / celda))
    return xe, ye, suave.T

In [ ]:
x, y = rm_utm.geometry.x.values, rm_utm.geometry.y.values
xe, ye, densidad = suavizar(x, y, celda=200, ancho=800)

fig, ax = plt.subplots(figsize=(7, 7))
umbral = np.percentile(densidad[densidad > 0], 70)
ax.imshow(np.ma.masked_less(densidad, umbral), extent=[xe[0], xe[-1], ye[0], ye[-1]], origin='lower', cmap='Blues')
ax.set_title('Región Metropolitana: concentración de siniestros')
ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Grilla de conteos y celdas "calientes": mucho más simple que un test estadístico formal,
# pero suficiente para ubicar dónde mirar primero. Una celda es "caliente" si su conteo
# está muy por sobre el promedio de sus vecinas (más de 2 desviaciones estándar).
celda = 400
ix = ((x - x.min()) // celda).astype(int)
iy = ((y - y.min()) // celda).astype(int)

grilla = pd.DataFrame({'ix': ix, 'iy': iy}).value_counts().rename('siniestros').reset_index()
promedio, desv = grilla['siniestros'].mean(), grilla['siniestros'].std()
grilla['caliente'] = grilla['siniestros'] > (promedio + 2 * desv)

print(f'Celdas totales: {len(grilla)} · celdas calientes: {grilla.caliente.sum()}')
grilla.sort_values('siniestros', ascending=False).head(8)

---
## 9. Ahora ustedes: mapa y puntos calientes para La Araucanía

Repitan los pasos 7 y 8, esta vez con la tabla `araucania` que armaron en la sección 6. Usen `CRS_METROS_ARAUCANIA` (no el de la RM) al proyectar, porque La Araucanía cae en otro huso UTM.

In [ ]:
# 9.1 Construyan el GeoDataFrame de La Araucanía en grados y su versión proyectada en metros
# (usen CRS_METROS_ARAUCANIA). Llamen a las variables araucania_geo y araucania_utm.

# TU CÓDIGO AQUÍ

In [ ]:
# 9.2 Reutilicen la función suavizar() ya definida arriba (no hace falta escribirla de nuevo)
# para calcular la densidad de La Araucanía y grafíquenla con imshow, como en la RM.

# TU CÓDIGO AQUÍ

In [ ]:
# 9.3 Calculen la grilla de conteos y las celdas calientes de La Araucanía (celda = 400 m,
# el mismo criterio de "más de 2 desviaciones estándar sobre el promedio").
# ¿Cuántas celdas calientes encontraron? Compárenlo con la Región Metropolitana.

# TU CÓDIGO AQUÍ

## 10. Comparación final entre las dos regiones
Con las tablas `rm` y `araucania` ya limpias, arma una comparación directa. Ajusta los nombres de columnas si en tu tabla `araucania` quedaron distintos.

In [ ]:
comparacion = pd.DataFrame({
    'Región Metropolitana': [
        len(rm), int(rm['fallecidos'].sum()),
        round(rm['fallecidos'].sum() / len(rm) * 100, 2),
        round((rm['tipo'] == 'Atropello').mean() * 100, 1),
    ],
    'La Araucanía': [
        len(araucania) if araucania is not None else None,
        int(araucania['fallecidos'].sum()) if araucania is not None else None,
        round(araucania['fallecidos'].sum() / len(araucania) * 100, 2) if araucania is not None else None,
        round((araucania['tipo'] == 'Atropello').mean() * 100, 1) if araucania is not None else None,
    ],
}, index=['siniestros', 'fallecidos', 'severidad (%)', '% atropellos'])

comparacion

## 11. Preguntas para el informe
Respondan estas preguntas en el informe breve, apoyándose en lo que calcularon en este notebook. No hace falta código adicional: son de interpretación.

1. **Validación.** ¿Qué tan cerca quedaron sus conteos de las cifras oficiales de CONASET? Si hay una diferencia grande, ¿a qué la atribuyen?
2. **Volumen contra severidad.** Nombren dos comunas o zonas con muchos siniestros y baja severidad, y dos con pocos siniestros y alta severidad. ¿Qué tienen en común las del segundo grupo?
3. **Patrón horario.** ¿A qué hora se concentran los siniestros en cada región? ¿Es la misma franja en ambas?
4. **Puntos calientes.** ¿Dónde aparecen las celdas calientes de cada región? Si tuvieran que priorizar una intervención, ¿por cuál empezarían y con qué criterio?
5. **De la hipótesis a la decisión.** Retomando la lámina 15 de la clase: elijan un hallazgo de su propio análisis y propongan un dato adicional (de movilidad, de cámaras, de flujo) que permitiría convertirlo en una decisión concreta de un plan de seguridad vial.
6. **Limitaciones.** Mencionen al menos dos límites del análisis que hicieron hoy (por ejemplo, la ventana de 48 horas, la falta de datos de exposición, o el tamaño de la celda que usaron para el mapa de calor).

**Entrega:** este notebook ejecutado de principio a fin, con las dos regiones completas, más un informe breve en Word con los gráficos y las seis respuestas.